# PhiState continuation journal

Use one Colab session for this experiment. On a connected L4 or T4 GPU, choose **Runtime → Run all** and authorize the Google account that owns the PhiState checkpoints. The journal reads the saved broad SFT run, checks the GPU, runs the optimizer comparison, and writes its results to Drive. A fresh comparison allows up to **500 additional optimizer updates per arm**, 1,000 total. An existing experiment keeps its recorded budget. Reopening a completed comparison shows its results.

The two arms start from the same checkpoint and sampler position. One keeps the original Adafactor policy. The other protects memory controls, normalization scales and biases from weight decay. Keeping the batches and data mixture fixed isolates that change. Training uses PyTorch on the GPU. A passing candidate can be used for native RFTMW C++ CPU chat.

**Save and stop.** Interrupt the training cell once and wait for its saved-stop message. To resume, run the journal again. The latest atomic save is the recovery point if Colab deletes the VM. The Drive path keeps the original project name `PhiState-L4` on either GPU.


## 1. Connect Drive and load the version used by this journal

The setup cell verifies the checkpoint and data locations before launching the worker. If Google asks for Drive access, finish the consent dialog. A credential error here occurs before training. Colab's Resources panel shows your current compute units and hourly rate.


In [ ]:
from pathlib import Path
import importlib.util
import json
import shutil
import subprocess
import sys
import tempfile
from google.colab import drive

MOUNT = Path('/content/drive')
ROOT = MOUNT / 'MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1'
SFT = ROOT / 'chat_recovery_v1/data'
SOURCE = ROOT / 'broad_continuation_v1/latest.pt'
OUT = ROOT / 'coherence_execution_v1'
if not (MOUNT / 'MyDrive').is_dir():
    try:
        drive.mount(str(MOUNT))
    except Exception as exc:
        raise RuntimeError('Drive setup stopped before training. Reconnect Colab, rerun this cell, select the Google account that owns the PhiState folder, and complete the Drive consent dialog.') from exc
required = [SOURCE, ROOT / 'corpus_repaired_v1/manifest.json', ROOT / 'corpus_repaired_v1/tokenizer.model',
            SFT / 'manifest.json', SFT / 'train.jsonl', SFT / 'val.jsonl', SFT / 'tokenizer.model']
missing_inputs = [str(path) for path in required if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError('The mounted account is missing these PhiState inputs:\n' + '\n'.join(missing_inputs))

PIN = '67c27c5ad55766f50aa54cedafae662be90d280a'
REPO = 'https://github.com/Anharmoniclabs/quantoniumoslinux.git'
CHECKOUT = Path('/content') / ('phistate_continuation_' + PIN[:12])
def git(repo, *args):
    return subprocess.check_output(['git', '-C', str(repo), *args], text=True).strip()

if not CHECKOUT.exists():
    stage = Path(tempfile.mkdtemp(prefix='phistate_setup_', dir='/content'))
    try:
        git(stage, 'init', '-q')
        git(stage, 'remote', 'add', 'origin', REPO)
        git(stage, 'config', 'remote.origin.promisor', 'true')
        git(stage, 'config', 'remote.origin.partialclonefilter', 'blob:none')
        git(stage, 'sparse-checkout', 'set', '--no-cone',
            '/runtime/experiments/phi_runtime_lab/', '/runtime/src/rftmw_native/')
        git(stage, 'fetch', '-q', '--depth=1', '--filter=blob:none', 'origin', PIN)
        git(stage, 'checkout', '--detach', '-q', 'FETCH_HEAD')
        stage.rename(CHECKOUT)
    finally:
        if stage.exists():
            shutil.rmtree(stage)
if git(CHECKOUT, 'rev-parse', 'HEAD') != PIN or git(CHECKOUT, 'status', '--porcelain', '--untracked-files=no'):
    raise RuntimeError('The runtime checkout differs from the pinned journal code.')

missing = [name for name in ('sentencepiece', 'numpy') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import torch
LAB = CHECKOUT / 'runtime/experiments/phi_runtime_lab'
# A new interpreter for each helper avoids stale imports after another notebook used this runtime.
def lab_python(script, *args):
    subprocess.run([sys.executable, '-u', '-c', script, *map(str, args)], cwd=LAB, check=True)

saved = SOURCE.parent / 'progress.json'
if saved.is_file():
    progress = json.loads(saved.read_text())
    print('Drive source:', SOURCE, '| saved SFT step:', progress.get('saved_sft_step'))
else:
    print('Drive source:', SOURCE)
print('Pinned code:', PIN)
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
lab_python("import sys; from continuation_journal import print_summary; print_summary(sys.argv[1])", ROOT)


## 2. Start or resume the comparison

To follow the full log from the Colab Terminal, use `tail -f /content/drive/MyDrive/Quantonium/PhiState-L4/whitespace_repair_v1/coherence_continuation_journal.log` once the worker starts.

The worker checks CPU/GPU numerical agreement before training. Its live log and Drive checkpoint summaries identify the active arm and the last saved step. A candidate must pass development checks for instructions, knowledge, arithmetic, and simple code while retaining its replay score. The chosen candidate then receives one separate final evaluation. Native export requires both qualification and numerical agreement with PyTorch.


In [ ]:
# A single interrupt requests a saved stop from this launcher.
import runpy
sys.path.insert(0, str(LAB))
controls = runpy.run_path(str(LAB / 'continuation_journal.py'), run_name='phistate_journal_controls')
previous_report = OUT / 'report.json'
previous_status = json.loads(previous_report.read_text()).get('status') if previous_report.is_file() else None
if previous_status not in {'qualified', 'failed_quality', 'failed_final'}:
    if not torch.cuda.is_available() or not hasattr(torch.optim, 'Adafactor'):
        raise RuntimeError('Select a CUDA GPU runtime with PyTorch Adafactor support, then run the journal again.')
JOURNAL_RUN_RESULT = {}
JOURNAL_RUN_RESULT = controls['run_or_resume'](ROOT, LAB, steps=500)
print('Comparison status:', JOURNAL_RUN_RESULT['status'])


## 3. Read the run and compare progress

The table reports saved updates and generated-answer pass rates per arm. The plots put answer loss and functional performance on separate axes. A drop in answer loss can coexist with an incorrect generated answer. The full results and examples are kept under `coherence_execution_v1`.


In [ ]:
lab_python("import sys; from continuation_journal import print_summary; print_summary(sys.argv[1])", ROOT)
report_path = OUT / 'report.json'
RESULT = json.loads(report_path.read_text()) if report_path.is_file() else {}
TERMINAL_RESULT = (globals().get('JOURNAL_RUN_RESULT', {}).get('status') == RESULT.get('status')
                   and RESULT.get('status') in {'qualified', 'failed_quality', 'failed_final'})
curves = {}
for arm in ('original', 'protected'):
    path = OUT / arm / 'metrics.jsonl'
    saved = OUT / arm / 'progress.json'
    if not path.is_file() or not saved.is_file():
        continue
    last_saved = json.loads(saved.read_text()).get('phase_step', 0)
    points = {}
    for line in path.read_text().splitlines():
        try:
            row = json.loads(line)
            step = int(row['phase_step'])
            if step <= last_saved:
                points[step] = (row['answer_validation']['val_answer_bpt'], row['quality']['macro_score'])
        except (KeyError, ValueError, TypeError):
            continue
    if points:
        curves[arm] = sorted(points.items())
if curves:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for arm, points in curves.items():
        xs = [step for step, _ in points]
        for index, axis in enumerate(axes):
            axis.plot(xs, [values[index] for _, values in points], marker='o', label=arm)
    for axis, label in zip(axes, ['Held-out answer bits/token', 'Development macro pass rate']):
        axis.set(xlabel='Additional optimizer updates', ylabel=label)
        axis.grid(alpha=.25)
        axis.legend()
    axes[1].set_ylim(-.025, 1.025)
    fig.tight_layout()
    plt.show()


## 4. Measure how the question changes the prediction

After the comparison ends, score up to 16 validation answers from each of the four sources under three conditions. The correct conversation, a different conversation from the same source, and an empty user message all precede the same answer tokens. The table shows how many additional bits per answer token the wrong or empty prompt costs. A positive value means the original question helped. Comparing the first eight answer tokens shows how strongly the question influences the start of a response.

This uses prepared validation rows. The generated-answer qualification and final evaluation are separate. The journal saves reports by checkpoint and data hash for reuse. Each row's scored count makes omitted pairs visible.


In [ ]:
if TERMINAL_RESULT:
    lab_python("""
import sys
from pathlib import Path
import torch
from chat_recovery_train import run_lock
from prompt_conditioning_eval import run
torch.set_num_threads(2)
root = Path(sys.argv[1])
out = root / 'coherence_execution_v1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
with run_lock(out / 'execution'):
    inputs = [('source', out/'source.pt')] + [(name, out/name/'latest.pt') for name in ('original','protected')]
    print('model       source         scored   wrong-true BPT   empty-true BPT  first8 wrong-true')
    for label, checkpoint in inputs:
        result = run(checkpoint, root/'chat_recovery_v1/data', out/'prompt_conditioning', device=device, limit=16)
        for name, entry in result['per_source'].items():
            measures = entry['comparisons']
            def mean_text(key):
                item = measures[key]
                return f"{item['mean']:+.4f}" if item is not None else 'unpaired'
            fields = [mean_text(k) for k in ('wrong_minus_correct_answer_bpt','empty_minus_correct_answer_bpt','wrong_minus_correct_first8_bpt')]
            scored = f"{entry['scored']}/{entry['selected']}"
            print(f"{label:<11} {name:<13} {scored:>6}   {fields[0]:>14}   {fields[1]:>14}   {fields[2]:>16}")
        print('Report:', result['report_path'])
""", ROOT)
else:
    print('Conditioning evaluation becomes available when both comparison arms have reached an outcome.')


## 5. Try a native response

A qualifying export gets one sampled C++ reply so **Run all** can finish. Edit the question and run this cell again to try another prompt. Set `INTERACTIVE` to True for chat with history. When qualification fails, the previous sections contain the actual development answers and the conditioning measurements.


In [ ]:
QUESTION = 'What is the difference between RAM and storage?'
INTERACTIVE = False
if TERMINAL_RESULT and RESULT.get('status') == 'qualified':
    import sys
    sys.path.insert(0, str(LAB))
    from chat_recovery_cpp import chat
    if Path(sys.modules['chat_recovery_cpp'].__file__).resolve().parent != LAB.resolve():
        raise RuntimeError('A different notebook imported another copy of the C++ chat module. Restart this notebook session before interactive chat.')
    chat(Path(RESULT['native_run']), prompt=None if INTERACTIVE else QUESTION, count=96,
         decode_config=dict(temperature=.7, top_p=.9, top_k=50, repetition_penalty=1.1, seed=20261031))
else:
    print('Native qualification status:', RESULT.get('status', 'pending'))
